In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:

from openai import OpenAI
openai_client = OpenAI()

In [3]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [4]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [5]:
# def llm(prompt):
#     response = openai_client.responses.create(
#         model="gpt-5.4-mini",
#         input=prompt
#     )
#     return response.output_text

In [6]:
# llm("Hey, what's up?")

In [7]:
question = "I just discovered the course. Can I join now?"


In [8]:
# context = """
# I just discovered the course. Can I still join?
# Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

# Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
# You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

# What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
# The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

# Cloud alternatives with GPU
# Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
# """

In [9]:
# prompt = f"""
# Your task is to answer questions from the course participants
# based on the provided context.

# Use the context to find relevant information and provide accurate
# answers. If the answer is not found in the context,
# respond with "I don't know."

# Question:
# {question}

# Context:
# {context}
# """

In [10]:
# answer = llm(prompt)
# print(answer)

In [11]:
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt)

In [12]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [13]:
documents[1342]

{'id': 'b75fa5280e',
 'course': 'machine-learning-zoomcamp',
 'section': 'Miscellaneous',
 'question': 'How do I install extra packages on Google Colab or Kaggle?',
 'answer': "In a notebook cell, prefix `pip install` with `!`:\n\n```bash\n!pip install tensorflow[and-cuda]==2.14\n```\n\nFor packages with extras you want installed silently, use `-q`:\n\n```bash\n!pip install -q xgboost==2.1.0\n```\n\nRestart the runtime if a package overrides one that's already imported (Runtime → Restart runtime)."}

In [14]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
index.search(question)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
 

In [16]:
# Adding filter to search
# index.search(question, filter_dict= {'course':'llm-zoomcamp'})

In [17]:
# # Select only 5 results
# search_results = index.search(
#     question, 
#     filter_dict={'course':'llm-zoomcamp'}, 
#     num_results=5
# )

In [18]:
# # We can boost records when we do search (ponderation)
# search_results = index.search(
#     question,
#     boost_dict= {'question':2.0, 'section':0.5}, 
#     filter_dict={'course':'llm-zoomcamp'}, 
#     num_results=5
# )

In [19]:
# def search(question):
# 	return index.search(
# 		question,
#     	boost_dict= {'question':2.0, 'section':0.5}, 
# 		filter_dict={'course':'llm-zoomcamp'},
# 		num_results=5
# 	)

In [20]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question':2.0, 'section':0.5}
    filter_dict = {'course':course}

    return index.search(
		question,
    	boost_dict=boost_dict, 
		filter_dict=filter_dict,
		num_results=5
	)

In [21]:
search_results = search(question)

In [22]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [23]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [24]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [25]:
# context = build_context(search_results)
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [26]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question, 
        context=context)
    return prompt.strip()

In [27]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [28]:
prompt = build_prompt(question, search_results)

In [29]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [30]:
# response = openai_client.responses.create(
#         model="gpt-5.4-mini",
#         input=prompt
#     )

In [31]:
# response.output_text

In [32]:
# print(response.model_dump_json(indent=2))

In [33]:
# response.output[0]

In [34]:
# response.output[0].content[0]

In [35]:
# response.output[0].content[0].text

In [36]:
# response.usage

In [37]:
# input_price = 0.75 / 1_000_000
# output_price = 4.50 / 1_000_000

# cost = (
#     response.usage.input_tokens * input_price +
#     response.usage.output_tokens * output_price
# )

# cost

In [38]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = openai_client.responses.create(
            model=model,
            input=message_history
        )
    return response.output_text

In [39]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [40]:
query = 'Is the homework mandatory?'
answer = rag(query)
print(answer)

I don’t know.


In [41]:
answer

'I don’t know.'

In [42]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a live cohort and pass the Capstone project. The certificate name will use the official name you set in your course profile.'

In [43]:
rag("can i get certificate if i missed homework")

'Yes — homework is not mandatory for the certificate. You need to pass the Capstone project to get the certificate.'